In [1]:
!module list 2>/dev/null || true
!nvidia-smi 2>/dev/null || true
!which mpirun 2>/dev/null || true
!which python3
%env HOROVOD_GPU_ALLREDUCE=NCCL
%env HOROVOD_GPU_ALLGATHER=NCCL
%env HOROVOD_GPU_BROADCAST=NCCL
%env NCCL_DEBUG=DEBUG

In [2]:
!nvidia-smi -L 2>/dev/null || echo 'No GPU or nvidia-smi available in current environment'

In [3]:
try:
    import horovod
    from horovod import run
    has_horovod = True
except (ImportError, ModuleNotFoundError):
    has_horovod = False
    run = None

def dummy(epochs=50, batch_size=8, numHiddenUnits=128, shrink=16, sample_limit=None):
    #----------------------------------------
    # Import packages
    #----------------------------------------
    import sys
    import os
    os.environ["TF_DISABLE_NVTX_RANGES"] = "1"
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    os.environ["NCCL_DEBUG"] = "WARN"
    import time
    import socket
    import math
    import numpy as np
    import xarray as xr
    try:
        import dask
        import dask.array as da
    except (ImportError, ModuleNotFoundError):
        pass
    try:
        import zarr as zr
    except (ImportError, ModuleNotFoundError):
        pass
    import pickle
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    from tensorflow import keras
    from tensorflow.keras import layers, models, losses
    from tensorflow.keras.regularizers import l1, l2
    from tensorflow.keras.optimizers import Optimizer
    
    # Set logging level to suppress unnecessary warnings
    tf.get_logger().setLevel(tf.compat.v1.logging.ERROR)
    
    #------------------------------------------------------------
    # Initialise Horovod (with single-process fallback)
    #------------------------------------------------------------
    try:
        import horovod.tensorflow.keras as hvd
        hvd.init()
        has_hvd = True
    except Exception as e:
        has_hvd = False
        class DummyHVD:
            def size(self): return 1
            def rank(self): return 0
            def local_rank(self): return 0
            def allreduce(self, val, name=None): return val
        hvd = DummyHVD()
    
    print('***hvd.size ', hvd.size(),' hvd.rank', hvd.rank(), 'hvd.local_rank() ', hvd.local_rank())
    
    # Horovod: pin GPU to be used to process local rank (one GPU per process)
    gpus = tf.config.experimental.list_physical_devices('GPU')
    print("Num GPUs Available: ", len(gpus))
    print(' gpus = ', gpus)
    if hvd.local_rank() == 0:
        print("Socket and len gpus = ", socket.gethostname(), len(gpus))
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    if gpus:
        gpu_to_use = gpus[hvd.local_rank() % len(gpus)] if has_hvd else gpus[0]
        try:
            tf.config.experimental.set_visible_devices(gpu_to_use, 'GPU')
        except Exception:
            pass

    #----------------------------------------------------------------------------------------------
    #
    # Build downscaling model using TensorFlow and Horovod - this version uses ResAFNO
    #
    #----------------------------------------------------------------------------------------------
    class FiLMLayer(layers.Layer):
        def __init__(self, channels: int, **kwargs):
            super().__init__(**kwargs)
            self.channels = int(channels)
            self.dense = layers.Dense(
                self.channels * 2,
                kernel_initializer="zeros",
                bias_initializer="zeros",
                name="film_dense",
            )
        def call(self, x, condition):
            gamma_beta = self.dense(condition)
            gamma, beta = tf.split(gamma_beta, num_or_size_splits=2, axis=-1)
            gamma = tf.reshape(gamma, [-1, 1, 1, self.channels])
            beta = tf.reshape(beta, [-1, 1, 1, self.channels])
            return x * (1.0 + gamma) + beta

    class AFNO2D(layers.Layer):
        def __init__(self, embed_dim: int, num_blocks: int = 8, sparsity_threshold: float = 0.01, **kwargs):
            super().__init__(**kwargs)
            self.embed_dim = int(embed_dim)
            self.num_blocks = int(num_blocks)
            self.block_size = int(embed_dim // num_blocks)
            self.sparsity_threshold = float(sparsity_threshold)
        def build(self, input_shape):
            scale = 0.02
            weight_shape = (self.num_blocks, self.block_size, self.block_size)
            bias_shape = (1, 1, self.num_blocks, self.block_size)
            for name, shape in (
                ("w1_real", weight_shape), ("w1_imag", weight_shape),
                ("b1_real", bias_shape), ("b1_imag", bias_shape),
                ("w2_real", weight_shape), ("w2_imag", weight_shape),
                ("b2_real", bias_shape), ("b2_imag", bias_shape),
            ):
                setattr(self, name, self.add_weight(name=name, shape=shape,
                    initializer=tf.random_normal_initializer(stddev=scale), trainable=True))
            super().build(input_shape)
        def _complex_mul(self, xr, xi, wr, wi):
            out_real = tf.einsum("...bi,bio->...bo", xr, wr) - tf.einsum("...bi,bio->...bo", xi, wi)
            out_imag = tf.einsum("...bi,bio->...bo", xr, wi) + tf.einsum("...bi,bio->...bo", xi, wr)
            return out_real, out_imag
        def _complex_softshrink(self, real, imag, lambd):
            magnitude = tf.sqrt(tf.square(real) + tf.square(imag))
            scale = tf.maximum(magnitude - lambd, 0.0) / tf.maximum(magnitude, tf.cast(1.0e-8, magnitude.dtype))
            return real * scale, imag * scale
        def _zero_mode_bias(self, bias, batch, frequency_count):
            zero = tf.broadcast_to(bias, [batch, 1, self.num_blocks, self.block_size])
            remainder = tf.zeros([batch, frequency_count - 1, self.num_blocks, self.block_size], dtype=zero.dtype)
            return tf.concat([zero, remainder], axis=1)
        def call(self, x):
            shape = tf.shape(x)
            batch, height, width = shape[0], shape[1], shape[2]
            reshaped = tf.reshape(x, [batch, height, width, self.num_blocks, self.block_size])
            packed = tf.transpose(reshaped, [0, 3, 4, 1, 2])
            norm_scale = tf.cast(tf.sqrt(tf.cast(height * width, tf.float32)), tf.complex64)
            spectrum = tf.signal.rfft2d(packed) / norm_scale
            spectrum = tf.transpose(spectrum, [0, 3, 4, 1, 2])
            spectrum_shape = tf.shape(spectrum)
            height_f, width_f = spectrum_shape[1], spectrum_shape[2]
            real = tf.reshape(tf.math.real(spectrum), [batch, height_f * width_f, self.num_blocks, self.block_size])
            imag = tf.reshape(tf.math.imag(spectrum), [batch, height_f * width_f, self.num_blocks, self.block_size])
            real, imag = self._complex_mul(real, imag, self.w1_real, self.w1_imag)
            real = real + self._zero_mode_bias(self.b1_real, batch, height_f * width_f)
            imag = imag + self._zero_mode_bias(self.b1_imag, batch, height_f * width_f)
            real, imag = self._complex_softshrink(real, imag, self.sparsity_threshold)
            real, imag = self._complex_mul(real, imag, self.w2_real, self.w2_imag)
            real = real + self._zero_mode_bias(self.b2_real, batch, height_f * width_f)
            imag = imag + self._zero_mode_bias(self.b2_imag, batch, height_f * width_f)
            real = tf.reshape(real, [batch, height_f, width_f, self.num_blocks, self.block_size])
            imag = tf.reshape(imag, [batch, height_f, width_f, self.num_blocks, self.block_size])
            output_spectrum = tf.complex(real, imag)
            output_spectrum = tf.transpose(output_spectrum, [0, 3, 4, 1, 2]) * norm_scale
            output = tf.signal.irfft2d(output_spectrum, fft_length=[height, width])
            output = tf.transpose(output, [0, 3, 4, 1, 2])
            output = tf.reshape(output, [batch, height, width, self.embed_dim])
            return tf.ensure_shape(output, [None, None, None, self.embed_dim])

    class AFNOResBlock(layers.Layer):
        def __init__(self, channels: int, num_blocks: int = 8, mlp_ratio: float = 2.0, sparsity_threshold: float = 0.01, **kwargs):
            super().__init__(**kwargs)
            self.channels = int(channels)
            self.num_blocks = int(num_blocks)
            self.mlp_ratio = float(mlp_ratio)
            self.sparsity_threshold = float(sparsity_threshold)
            self.norm1 = layers.LayerNormalization(epsilon=1.0e-5)
            self.afno = AFNO2D(self.channels, num_blocks=self.num_blocks, sparsity_threshold=self.sparsity_threshold)
            self.film = FiLMLayer(self.channels)
            self.norm2 = layers.LayerNormalization(epsilon=1.0e-5)
            hidden = int(self.channels * self.mlp_ratio)
            self.conv1 = layers.Conv2D(hidden, 3, padding="same", activation="gelu")
            self.conv2 = layers.Conv2D(self.channels, 3, padding="same")
        def call(self, x, condition=None):
            x = x + self.afno(self.norm1(x))
            if condition is not None:
                x = self.film(x, condition)
            refinement = self.conv2(self.conv1(self.norm2(x)))
            return x + 0.2 * refinement

    class ProgressiveUpsampleBlock(layers.Layer):
        def __init__(self, in_channels: int, out_channels: int, **kwargs):
            super().__init__(**kwargs)
            self.in_channels = int(in_channels)
            self.out_channels = int(out_channels)
            self.upsample = layers.UpSampling2D(size=2, interpolation="bilinear")
            self.conv_in = layers.Conv2D(self.out_channels, 3, padding="same", activation="gelu")
            self.film = FiLMLayer(self.out_channels)
            self.conv_refine = layers.Conv2D(self.out_channels, 3, padding="same", activation="gelu")
            self.conv_out = layers.Conv2D(self.out_channels, 3, padding="same")
        def call(self, x, condition=None):
            h = self.conv_in(self.upsample(x))
            if condition is not None:
                h = self.film(h, condition)
            residual = self.conv_out(self.conv_refine(h))
            return h + 0.2 * residual

    class CoarseConsistencyProjection(layers.Layer):
        def __init__(self, shrink: int, **kwargs):
            super().__init__(**kwargs)
            self.shrink = int(shrink)
        def call(self, inputs):
            values, coarse_sst = inputs
            values = tf.cast(values, tf.float32)
            coarse_sst = tf.cast(coarse_sst, values.dtype)
            shape = tf.shape(values)
            batch, height, width, channels = shape[0], shape[1], shape[2], shape[3]
            coarse_height = height // self.shrink
            coarse_width = width // self.shrink
            reshaped_values = tf.reshape(values, [batch, coarse_height, self.shrink, coarse_width, self.shrink, channels])
            block_mean = tf.reduce_mean(reshaped_values, axis=[2, 4])
            correction = coarse_sst - block_mean
            correction = tf.repeat(tf.repeat(correction, self.shrink, axis=1), self.shrink, axis=2)
            return values + correction

    class ResAFNOTrainer(models.Model):
        """Subclassed model enabling standard model.fit with masked MSE and gradient clipping."""
        def __init__(self, core_model, shrink=16, **kwargs):
            super().__init__(**kwargs)
            self.core_model = core_model
            self.shrink = shrink
            self.loss_tracker = tf.keras.metrics.Mean(name="loss")
            self.mae_tracker = tf.keras.metrics.MeanAbsoluteError(name="mae")
        def call(self, inputs, training=False):
            return self.core_model(inputs, training=training)
        @property
        def metrics(self):
            return [self.loss_tracker, self.mae_tracker]
        def train_step(self, data):
            x, y = data
            with tf.GradientTape() as tape:
                y_pred = self.core_model(x, training=True)
                mask = tf.cast(tf.math.is_finite(y) & (tf.math.abs(y) > 1e-7), tf.float32)
                diff_sq = tf.square(y_pred - y) * mask
                loss = tf.reduce_sum(diff_sq) / tf.maximum(tf.reduce_sum(mask), 1.0)
                if self.losses:
                    loss += tf.add_n(self.losses)
            grads = tape.gradient(loss, self.trainable_variables)
            grads, _ = tf.clip_by_global_norm(grads, 1.0)
            self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
            self.loss_tracker.update_state(loss)
            self.mae_tracker.update_state(y, y_pred, sample_weight=mask)
            return {"loss": self.loss_tracker.result(), "mae": self.mae_tracker.result()}
        def test_step(self, data):
            x, y = data
            y_pred = self.core_model(x, training=False)
            mask = tf.cast(tf.math.is_finite(y) & (tf.math.abs(y) > 1e-7), tf.float32)
            diff_sq = tf.square(y_pred - y) * mask
            loss = tf.reduce_sum(diff_sq) / tf.maximum(tf.reduce_sum(mask), 1.0)
            self.loss_tracker.update_state(loss)
            self.mae_tracker.update_state(y, y_pred, sample_weight=mask)
            return {"loss": self.loss_tracker.result(), "mae": self.mae_tracker.result()}

    def SRDN_ResAFNO_v4(numHiddenUnits, numResponses, numFeatures, numLats, numLongs, shrink):
        coarse_shape = (int(numLats // shrink), int(numLongs // shrink), numFeatures)
        inputs = layers.Input(shape=coarse_shape, name="coarse_sst")
        coarse_sst = inputs

        coarse_skip = layers.UpSampling2D(size=int(shrink), interpolation="bilinear", name="physical_coarse_skip")(coarse_sst)
        cond_emb = layers.GlobalAveragePooling2D(name="cond_gap")(coarse_sst)
        cond_emb = layers.Dense(numHiddenUnits, activation="gelu", name="cond_mlp_1")(cond_emb)
        cond_emb = layers.Dense(numHiddenUnits, activation="gelu", name="cond_mlp_2")(cond_emb)

        x = layers.Conv2D(numHiddenUnits, 3, padding="same", activation="gelu", name="stem_conv")(coarse_sst)
        trunk_blocks = 6
        num_freq_blocks = 8
        for i in range(trunk_blocks):
            x = AFNOResBlock(
                channels=numHiddenUnits,
                num_blocks=num_freq_blocks,
                mlp_ratio=2.0,
                sparsity_threshold=0.01,
                name=f"res_afno_block_{i+1}",
            )(x, condition=cond_emb)

        upsample_stages = int(np.log2(shrink))
        channels_schedule = [128, 96, 64, 48][:upsample_stages]
        current_channels = numHiddenUnits
        for s, out_ch in enumerate(channels_schedule):
            x = ProgressiveUpsampleBlock(in_channels=current_channels, out_channels=out_ch, name=f"upsample_stage_{s+1}")(x, condition=cond_emb)
            current_channels = out_ch

        x = layers.Conv2D(32, 3, padding="same", activation="gelu", name="head_conv1")(x)
        residual = layers.Conv2D(numResponses, 3, padding="same", kernel_initializer="zeros", bias_initializer="zeros", name="head_conv_detail")(x)
        values = layers.Add(name="residual_plus_coarse")([residual, coarse_skip])
        outputs = CoarseConsistencyProjection(shrink, name="coarse_consistency_projection")([values, coarse_sst])

        core_model = models.Model(inputs=inputs, outputs=outputs, name="ResAFNO_Core")
        model = ResAFNOTrainer(core_model, shrink=shrink)

        # Horovod: adjust learning rate based on number of GPUs
        opt = tf.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999)
        if has_hvd:
            opt = hvd.DistributedOptimizer(opt)

        model.compile(optimizer=opt)
        print(core_model.summary()) if hvd.rank() == 0 else None
        return model

    t0 = [0]*hvd.size()
    t0[hvd.rank()] = time.time()

    # Set total number of images, training epoch size, test data size
    Total_images = 13149 # for SST daily data
    Epoch_size = 10560   # approx. 80%
    Test_size  = Total_images - Epoch_size # 2589
    if sample_limit is not None:
        Total_images = min(Total_images, sample_limit)
        Epoch_size = max(1, int(Total_images * 0.8))
        Test_size = max(1, Total_images - Epoch_size)

    numResponses = 1
    numFeatures  = 1
    numLats        = 512 
    numLongs       = 512

    if hvd.rank() == 0:
        print ('*** rank = ', hvd.rank(),' Epoch size = ', Epoch_size)
        print ('*** rank = ', hvd.rank(),' Test_size = ', Test_size)
        print ('*** rank = ', hvd.rank(),' Batch size = ', batch_size)
        print ('*** rank = ', hvd.rank(),' numHiddenUnits = ', numHiddenUnits)
        print ('*** rank = ', hvd.rank(),' numResponses= ', numResponses)
        print ('*** rank = ', hvd.rank(),' numFeatures = ', numFeatures)
        print ('*** rank = ', hvd.rank(),' numLats = ', numLats)
        print ('*** rank = ', hvd.rank(),' numLongs = ', numLongs)
        print ('*** rank = ', hvd.rank(),' shrink = ', shrink)

    #----------------------------------------------------------------------------------------------
    #
    # 	Open the input data files using xarray
    #
    #----------------------------------------------------------------------------------------------
    data_candidates = [
        "/g/data/sd82/sst_stand_10km_OFAM_historical_Australia_lon_interp.nc",
        "/esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling/sst_10km_OFAM_historical_Australia.nc",
        "../sst_10km_OFAM_historical_Australia.nc",
        "./sst_10km_OFAM_historical_Australia.nc"
    ]
    data_path = None
    for p in data_candidates:
        if os.path.exists(p):
            data_path = p
            break
    if data_path is None:
        raise FileNotFoundError(f"Could not find SST dataset. Checked: {data_candidates}")
    
    print(f"Opening dataset: {data_path}")
    ds = xr.open_dataset(data_path, decode_times=False)
    sst = ds["temp"]
    if "st_ocean" in sst.dims and sst.sizes["st_ocean"] == 1:
        sst = sst.squeeze("st_ocean")
    print("Shape after squeeze:", sst.shape)
    
    # Mean and standard deviation for normalization
    if "temp_mean" in ds and "temp_std" in ds:
        mean_val = float(ds["temp_mean"].values)
        std_val = float(ds["temp_std"].values)
    else:
        mean_val = 20.658
        std_val = 8.520

    elapsed_time = time.time() - t0[hvd.rank()]
    print ('*** rank = ', hvd.rank(),' Dataset Initialise Elapsed Time (sec) = ', elapsed_time)

    #----------------------------------------------------------------------------------------------
    #
    # Horovod: Split the train and test data across multiple processors   
    #
    #----------------------------------------------------------------------------------------------
    istart = int(hvd.rank()*Epoch_size/hvd.size())
    istop  = int((hvd.rank()+1)*Epoch_size/hvd.size())
    i_test_start = int(hvd.rank()*Test_size/hvd.size()+Epoch_size)
    i_test_stop  = int((hvd.rank()+1)*Test_size/hvd.size()+Epoch_size)
    if i_test_stop > Total_images:
        i_test_stop = Total_images

    print ('*** rank = ', hvd.rank(),' istart = ', istart, ' istop = ', istop)
    print ('*** rank = ', hvd.rank(),' i_test_start = ', i_test_start, ' i_test_stop = ', i_test_stop)

    t0[hvd.rank()] = time.time()
    # Read slices into float32 array
    x_tr_raw = np.asarray(sst[istart:istop, :, :].values, dtype=np.float32)
    x_te_raw = np.asarray(sst[i_test_start:i_test_stop, :, :].values, dtype=np.float32)
    
    # Standardize data if values are Celsius
    if np.nanmean(x_tr_raw) > 5.0:
        x_tr_n = np.where(np.isnan(x_tr_raw), 0.0, (x_tr_raw - mean_val) / std_val).astype(np.float32)
        x_te_n = np.where(np.isnan(x_te_raw), 0.0, (x_te_raw - mean_val) / std_val).astype(np.float32)
    else:
        x_tr_n = np.nan_to_num(x_tr_raw, nan=0.0).astype(np.float32)
        x_te_n = np.nan_to_num(x_te_raw, nan=0.0).astype(np.float32)

    x_tr_n = np.expand_dims(x_tr_n, axis=3)
    x_te_n = np.expand_dims(x_te_n, axis=3)

    # Downscale experiments start with low resolution data created by AveragePooling2D
    x_train = tf.keras.layers.AveragePooling2D(pool_size=(shrink, shrink), padding='same')(x_tr_n)
    x_test  = tf.keras.layers.AveragePooling2D(pool_size=(shrink, shrink), padding='same')(x_te_n)

    y_train = x_tr_n
    y_test  = x_te_n

    print(' Training data shapes = ', x_train.shape, y_train.shape)
    print(' Test data shapes = ', x_test.shape, y_test.shape)

    elapsed_time = time.time() - t0[hvd.rank()]
    print ('*** rank = ', hvd.rank(),' Dataset Read Elapsed Time (sec) = ', elapsed_time)

    # Callbacks
    callbacks = []
    if has_hvd:
        callbacks.append(hvd.callbacks.BroadcastGlobalVariablesCallback(0))
        callbacks.append(hvd.callbacks.MetricAverageCallback())
    if hvd.rank() == 0:
        os.makedirs('./checkpoints_resafno', exist_ok=True)
        callbacks.append(tf.keras.callbacks.ModelCheckpoint('./checkpoints_resafno/checkpoint-{epoch}.h5', monitor='val_loss', save_best_only=True, save_weights_only=True))

    # Build the ResAFNO model
    model = SRDN_ResAFNO_v4(numHiddenUnits, numResponses, numFeatures, numLats, numLongs, shrink)

    # Train the model
    t0[hvd.rank()] = time.time()
    if has_hvd:
        hvd.allreduce([0], name="Barrier")
    print ('*** rank = ', hvd.rank(),' Train model')

    history = model.fit(
        x_train, y_train,
        batch_size=batch_size,
        callbacks=callbacks,
        epochs=epochs,
        verbose=2,
        validation_data=(x_test, y_test)
    )
    if hvd.rank() == 0:
        print(history.history)

    elapsed_time = time.time() - t0[hvd.rank()]
    print ('*** rank = ', hvd.rank(),' Total Training Elapsed Time (sec) = ', elapsed_time)
    return model, history, (x_test, y_test, mean_val, std_val)


In [4]:
%%time
# Run training function
# In multi-GPU Horovod: run(dummy, use_mpi=True, np=num_gpus)
# In interactive Jupyter session: dummy(epochs=5, batch_size=8)
if has_horovod and 'run' in locals() and run is not None:
    import tensorflow as tf
    num_gpus = len(tf.config.experimental.list_physical_devices('GPU'))
    if num_gpus > 1:
        model, history, test_data = run(dummy, use_mpi=True, np=num_gpus)
    else:
        model, history, test_data = dummy(epochs=5, batch_size=8, sample_limit=64)
else:
    model, history, test_data = dummy(epochs=5, batch_size=8, sample_limit=64)

TensorFlow version: 2.15.1
***hvd.size  1  hvd.rank 0 hvd.local_rank()  0
Num GPUs Available:  0
 gpus =  []
Socket and len gpus =  login03 0
*** rank =  0  Epoch size =  51
*** rank =  0  Test_size =  13
*** rank =  0  Batch size =  8
*** rank =  0  numHiddenUnits =  128
*** rank =  0  numResponses=  1
*** rank =  0  numFeatures =  1
*** rank =  0  numLats =  512
*** rank =  0  numLongs =  512
*** rank =  0  shrink =  16
Opening dataset: /esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling/sst_10km_OFAM_historical_Australia.nc
Shape after squeeze: (13149, 512, 512)
*** rank =  0  Dataset Initialise Elapsed Time (sec) =  5.26577353477478
*** rank =  0  istart =  0  istop =  51
*** rank =  0  i_test_start =  51  i_test_stop =  64
 Training data shapes =  (51, 32, 32, 1) (51, 512, 512, 1)
 Test data shapes =  (13, 32, 32, 1) (13, 512, 512, 1)
*** rank =  0  Dataset Read Elapsed Time (sec) =  0.42474842071533203
Model: "ResAFNO_Core"
____________________________________________

In [5]:
import xarray as xr
import os
for path in [
    "/g/data/sd82/sst_stand_10km_OFAM_historical_Australia_lon_interp.nc",
    "/esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling/sst_10km_OFAM_historical_Australia.nc",
    "../sst_10km_OFAM_historical_Australia.nc",
    "./sst_10km_OFAM_historical_Australia.nc"
]:
    if os.path.exists(path):
        ds = xr.open_dataset(path, decode_times=False)
        print("Loaded Dataset from:", path)
        print(ds)
        print("\nVariables:")
        print(list(ds.variables))
        break

Loaded Dataset from: /esi/project/niwa03712/rampaln/PUBLICATIONS/2026/SSTDownscaling/sst_10km_OFAM_historical_Australia.nc
<xarray.Dataset> Size: 14GB
Dimensions:   (Time: 13149, st_ocean: 1, yt_ocean: 512, xt_ocean: 512)
Coordinates:
  * Time      (Time) float64 105kB 0.5 1.5 2.5 ... 1.315e+04 1.315e+04 1.315e+04
  * st_ocean  (st_ocean) float64 8B 2.5
  * xt_ocean  (xt_ocean) float64 4kB 107.3 107.4 107.6 ... 158.2 158.4 158.4
  * yt_ocean  (yt_ocean) float64 4kB -52.95 -52.85 -52.75 ... -2.05 -1.95 -1.85
Data variables:
    temp      (Time, st_ocean, yt_ocean, xt_ocean) float32 14GB ...
Attributes:
    filename:       TMP/ocean_ofam_1979_01.nc.0000
    NumFilesInSet:  720
    title:          jra_55_1979
    grid_type:      regular
    history:        Fri May 29 16:42:18 2026: ncks -O -d xt_ocean,43,554 sst_...
    NCO:            netCDF Operators version 5.0.5 (Homepage = http://nco.sf....

Variables:
['Time', 'st_ocean', 'temp', 'xt_ocean', 'yt_ocean']


In [6]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# Predict on test sample
x_test, y_test, mean_val, std_val = test_data
pred = model.predict(x_test[:1])

# Denormalize to Celsius
target_c = y_test[0, :, :, 0] * std_val + mean_val
pred_c   = pred[0, :, :, 0] * std_val + mean_val
coarse_up = tf.image.resize(x_test[:1], (512, 512), method="bilinear").numpy()[0, :, :, 0] * std_val + mean_val

mask = np.abs(y_test[0, :, :, 0]) > 1e-7
target_c = np.where(mask, target_c, np.nan)
pred_c = np.where(mask, pred_c, np.nan)
coarse_up = np.where(mask, coarse_up, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
im0 = axes[0].imshow(np.ma.masked_invalid(coarse_up), origin="lower", cmap="turbo")
axes[0].set_title("Low-Resolution Coarse Input (Bilinear)")
plt.colorbar(im0, ax=axes[0], label="SST (°C)")

im1 = axes[1].imshow(np.ma.masked_invalid(pred_c), origin="lower", cmap="turbo")
axes[1].set_title("ResAFNO Downscaled Prediction")
plt.colorbar(im1, ax=axes[1], label="SST (°C)")

im2 = axes[2].imshow(np.ma.masked_invalid(target_c), origin="lower", cmap="turbo")
axes[2].set_title("High-Resolution Target (OFAM)")
plt.colorbar(im2, ax=axes[2], label="SST (°C)")

plt.tight_layout()
plt.show()


1/1 [==============================] - 1s 1s/step
